In [40]:
NEO4J_URI="neo4j+s://2af643bb.databases.neo4j.io"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="1VcAM3_5lLyneDB2V7QTc09M0k8bugawY19gx9mHKB0"
AURA_INSTANCEID="2af643bb"
AURA_INSTANCENAME="Free instance"
# given in a text file while downlaoding the neo4j

In [41]:
import os 

os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [42]:
from langchain_community.graphs import Neo4jGraph
graph = Neo4jGraph(url=NEO4J_URI,username=NEO4J_USERNAME,password=NEO4J_PASSWORD)
graph

In [43]:
## dataset movies  
# 'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv'

movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""

movie_query

"\nLOAD CSV WITH HEADERS FROM\n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row\n\nMERGE(m:Movie{id:row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\nFOREACH (director in split(row.director, '|') | \n    MERGE (p:Person {name:trim(director)})\n    MERGE (p)-[:DIRECTED]->(m))\nFOREACH (actor in split(row.actors, '|') | \n    MERGE (p:Person {name:trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m))\nFOREACH (genre in split(row.genres, '|') | \n    MERGE (g:Genre {name:trim(genre)})\n    MERGE (m)-[:IN_GENRE]->(g))\n\n\n"

In [44]:
graph.query(movie_query)

[]

In [45]:
graph.refresh_schema()
print(graph.schema)

Node properties:
CEO {DOB: INTEGER, name: STRING, BA: STRING}
Company {name: STRING}
student {DOB: INTEGER, name: STRING, BA: STRING}
country {name: STRING}
person {name: STRING, born: INTEGER}
Movie {release: INTEGER, title: STRING, id: STRING, released: DATE, imdbRating: FLOAT}
User {name: STRING, city: STRING, userId: INTEGER, age: INTEGER}
Post {postId: INTEGER, content: STRING, timestamp: DATE_TIME}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:student)-[:LIVED_IN]->(:country)
(:person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:POSTED]->(:Post)
(:User)-[:FRIEND]->(:User)
(:User)-[:LIKES]->(:User)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


# creating graphquery chain 

In [46]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model_name = "Meta-Llama/Llama-4-Scout-17b-16e-Instruct" , groq_api_key = groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000237F8F94680>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000237F902FC50>, model_name='Meta-Llama/Llama-4-Scout-17b-16e-Instruct', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [47]:
from langchain.chains import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True  # <-- required flag
)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x00000237F8E5D3A0>, cypher_generation_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000237F8F94680>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000237F902FC50>, model_name=

In [48]:
response = chain.invoke(
    {"query" : "who was the director of the movie casino"}
)
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:

MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'Casino'})
RETURN p.name

Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.


{'query': 'who was the director of the movie casino',
 'result': 'Martin Scorsese was the director of the movie casino.'}

# prompting strategies 

In [1]:
# next module